In [1]:
# Multi class classification problem
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head() # 34 features , 1 category (class)

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [5]:
X = df.drop("Class",axis=1)
y = df["Class"]

In [6]:
df["Class"].unique() # There are 7 unique categories in the Class column

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
# In our ANN, the output layer will have 7 neurons because in multi-class classification,
# the number of output neurons is equal to the number of classes (categories)

In [8]:
# When we build an ANN for regression and classification problems, we generally use 1–2 hidden layers because for most basic problems, simple models are enough
# In hidden layers, we often start with the number of neurons as powers of 2 (e.g., 16, 32, 64)
# If the ANN is underfitting, we can increase the number of neurons in the hidden layers

In [9]:
encoder = LabelEncoder() # Used to convert categorical data into numbers
y = encoder.fit_transform(y)

In [10]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
scaler = StandardScaler() # Used to scale numerical data

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## ANN

In [13]:
# Converting Data into Tensors

# It is not always mandatory to convert y_train into y_train.values (same for y_test) because PyTorch can directly handle pandas Series or 
# NumPy arrays while creating tensors.

# In ANN, when performing multi-class classification, we use CrossEntropyLoss as the loss function
# It expects the target values (y) to be in integer (long) datatype representing class indices(numeric labels for categories (0, 1, 2))
# Therefore, we convert y_train and y_test to torch.long

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [14]:
# Creating Datasets
# TensorDataset is a PyTorch class that wraps input and target tensors together so they can be accessed as pairs and used with a DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [15]:
# Creating DataLoader(DataLoader class work is to define how the data will be loaded fro training)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [16]:
# In multi-class classification using ANN, we have: Input layer → Hidden layers → Output layer
# In hidden layers, we generally use Linear + ReLU activation functions.
# We have learned that Softmax is applied in the output layer to get class probabilities.
# However, in PyTorch, when we use CrossEntropyLoss as the loss function, it internally applies Softmax to the model outputs and then computes the loss.
# Therefore, we should NOT apply Softmax manually in the output layer, because it is already handled by CrossEntropyLoss.


# In multi-class classification, Softmax activation function is used in the following cases:

# a) When we need to calculate actual probabilities:
# The output layer (Linear) gives logits (raw values), e.g., [0, 0.6, 0.3, 2, 6, 7, 2]
# These values:
# - Do not sum to 1
# - Can be negative or greater than 1
# Softmax converts these logits into probabilities:
# - Values between 0 and 1
# - Sum of all values = 1

# b) When we are NOT using CrossEntropyLoss:
# If we use some other loss function, then we need to apply Softmax manually to convert logits into probabilities.

In [17]:
# Build our Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            # 2nd Hidden Layer
            nn.Linear(64, 64),
            nn.ReLU(),
            # Output layer
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [18]:
model = ANN()

# loss & optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [19]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.6631334760914678
epoch = 2/100, loss = 1.0941362717877263
epoch = 3/100, loss = 0.7526804688184158
epoch = 4/100, loss = 0.5629383584727412
epoch = 5/100, loss = 0.46356641274431476
epoch = 6/100, loss = 0.3933959616267163
epoch = 7/100, loss = 0.35945321295572363
epoch = 8/100, loss = 0.31629977083724475
epoch = 9/100, loss = 0.29918445970701135
epoch = 10/100, loss = 0.2742657123700432
epoch = 11/100, loss = 0.2439358393135278
epoch = 12/100, loss = 0.22858461867208066
epoch = 13/100, loss = 0.21381946669324584
epoch = 14/100, loss = 0.21115206797485767
epoch = 15/100, loss = 0.19606940513071808
epoch = 16/100, loss = 0.18828312534353006
epoch = 17/100, loss = 0.1781763097514277
epoch = 18/100, loss = 0.17242338832305826
epoch = 19/100, loss = 0.16741263607273932
epoch = 20/100, loss = 0.16190014423235602
epoch = 21/100, loss = 0.15997507747100748
epoch = 22/100, loss = 0.1499537054611289
epoch = 23/100, loss = 0.14310857443057973
epoch = 24/100, loss = 0.1420

In [21]:
# Evaluate
model.eval()

# Process:
# In the output layer, we have 7 neurons
# Each neuron represents a class
# The neuron with the highest value (highest probability after Softmax) is selected as the predicted class

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # The output layer (Linear) gives logits (raw values), e.g., [0, 0.6, 0.3, 2, 6, 7, 2] -- 7 values
                            # These values: Do not sum to 1 and Can be negative or greater than 1
        
        max_value, predicted = torch.max(outputs, 1) # Looks at all 7 values and finds the maximum value
                                                     # max_Value stores the maximum value and predicted stores the index of max_value

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch

print("accuracy: ", correct/total * 100)

accuracy:  95.0
